# 01 — Data loading & EDA

Exploration of the A/B email experiment with behavioral-science nudges.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import GROUP_LABELS, load_data

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (10, 5)

## Loading and validation

In [ ]:
df = load_data(ROOT / 'data' / 'datos_prueba_tecnica.csv')
print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
print(f'Null values: {df.isna().sum().sum()}')
df.head()

In [ ]:
assert len(df) == 5000
assert df['iid'].nunique() == 5000
assert set(df['grupo'].cat.categories) == {'ctrl', 'trat1', 'trat2'}
print('Basic validation OK')

## Randomization balance by group

In [ ]:
group_counts = df['grupo'].value_counts().sort_index()
group_share = (group_counts / len(df) * 100).round(1)
balance = pd.DataFrame({'N': group_counts, '%': group_share})
balance.index = balance.index.map(GROUP_LABELS)
balance

## Outcome rates by group

In [ ]:
rates = (
    df.groupby('grupo', observed=True)[['or', 'ctor']]
    .mean()
    .rename(columns={'or': 'open_rate', 'ctor': 'click_rate'})
)
rates['ctor_given_open'] = (
    df[df['or'] == 1].groupby('grupo', observed=True)['ctor'].mean()
)
rates.index = rates.index.map(GROUP_LABELS)
rates.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(
    axes,
    ['or', 'ctor'],
    ['Open rate (or)', 'Click rate (ctor)'],
):
    sns.barplot(data=df, x='grupo', y=metric, errorbar=('ci', 95), ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Grupo')
    ax.set_ylabel('Proportion')
plt.tight_layout()
plt.show()

## Covariate distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df['edad'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Age')
sns.histplot(df['inve'], kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Investment')
sns.countplot(data=df, x='uso_app', hue='grupo', ax=axes[1, 0])
axes[1, 0].set_title('App usage by group')
sns.countplot(data=df, x='formacion', hue='grupo', ax=axes[1, 1])
axes[1, 1].set_title('Education by group')
plt.tight_layout()
plt.show()

## Are the covariates balanced across groups?

In [ ]:
covariates = ['edad', 'inve', 'sexo', 'uso_app', 'tarjeta_debito']
balance_tests = []
for col in covariates:
    groups = [g[col].values for _, g in df.groupby('grupo', observed=True)]
    if col in ['edad', 'inve']:
        stat, p = stats.f_oneway(*groups)
        test = 'ANOVA'
    else:
        table = pd.crosstab(df['grupo'], df[col])
        stat, p, _, _ = stats.chi2_contingency(table)
        test = 'Chi-cuadrado'
    balance_tests.append({'variable': col, 'test': test, 'p_value': p})
balance_df = pd.DataFrame(balance_tests).round(4)
balance_df

**EDA takeaway:** The three arms are reasonably balanced. The treatments show a clear increase in open rate and click rate versus control; `trat2` appears to beat `trat1` on clicks.